# 01 — LCZ-Stratified Time-Series Visualization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/local/01_lcz_time_series.en.ipynb)

**Learning objective**: learn how LCZ4py assigns each weather station to a Local Climate Zone (LCZ) class and uses that class as a grouping variable to visualize air-temperature time series — the foundational step for every station-based ("local") urban-climate analysis in this package, from anomaly detection to Urban Heat Island (UHI) intensity.

## Why LCZ-stratified time series?

The Local Climate Zone scheme (Stewart & Oke, 2012) classifies urban and natural landscapes into 17 standardized categories (LCZ 1–10: built types, from compact high-rise to sparsely built; LCZ 11(A)–17(G): land-cover types, from dense trees to water) based on physical properties that control the local surface energy balance — sky view factor, building/impervious surface fraction, surface admittance, and anthropogenic heat output. Two stations 500 m apart can sit in very different LCZs (e.g. a compact midrise courtyard vs. a park) and therefore experience systematically different thermal regimes, even though a conventional "urban vs. rural" station network would lump them together or, worse, discard the comparison entirely for lack of a shared reference frame.

**LCZ4py's `local/` module family exists to make this stratification automatic.** Given a classified LCZ raster (e.g. from `lcz_get_map`) and a table of station observations with latitude/longitude, `lcz_ts` performs a point-in-raster spatial join: each station is assigned the LCZ class of the pixel it falls on. From that point forward, every station carries an LCZ label, and time series can be grouped, compared, and faceted by urban form category instead of treated as an undifferentiated bag of sensors. This is exactly the workflow described for the R package this port is based on — Anjos, M. et al. (2025), *Scientific Reports*, https://www.nature.com/articles/s41598-025-92000-0 — and it is the prerequisite step for every downstream `local/` function: temperature anomalies (notebook 02), UHI intensity (03), spatial interpolation (04–05), and derived climate metrics (06–07).

**Temporal aggregation (`time_freq`)** controls the resolution of the averaging window before plotting:
- `"hour"` — preserves the diurnal cycle; best for looking at daytime/nighttime contrasts (e.g. nocturnal UHI).
- `"day"` — smooths hourly noise, useful for multi-month or multi-year overviews.
- `"month"` — reveals the seasonal cycle, useful for year-over-year comparison.

**`facet_plot='lcz'`** (used with `plot_type="heatmap"`) puts one row per LCZ class on a shared time axis, making it visually obvious whether, say, compact urban classes (LCZ 2–3) run systematically warmer at night than open/natural classes (LCZ 11–14) — the signature diurnal fingerprint of the urban heat island. The `by=` parameter offers a complementary decomposition — splitting into subplots by `"season"`, `"year"`, `"weekday"`/`"weekend"`, or a special `"daylight"` mode that shades daytime windows directly on a single unified chart.

In [1]:
!pip -q install "LCZ4py[all] @ git+https://github.com/ByMaxAnjos/LCZ4py.git"


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Applications/QGIS.app/Contents/MacOS/bin/python3 -m pip install --upgrade pip
ERROR: Package 'lcz4py' requires a different Python: 3.9.5 not in '>=3.10'


## 1. Set up the shared data pipeline: Berlin LCZ map + station observations

This is the first notebook in the `local/` series, so it sets up the two artifacts every later `local/` notebook reuses:
1. **`lcz_map`** — the classified LCZ GeoTIFF for Berlin, downloaded once with `lcz_get_map` (global map, Demuzere et al. 2022) and cached locally.
2. **`df`** — hourly air-temperature observations from 23 Berlin-area stations (`lcz_data_berlin.csv`, 2019–2020), loaded straight from the GitHub raw URL so it works in any fresh Colab session with no upload step.

Columns in `lcz_data_berlin.csv`: `date` (datetime), `station` (station id), `airT` (air temperature, °C), `Latitude`, `Longitude`.

In [2]:
import pandas as pd
from LCZ4py import lcz_get_map
from LCZ4py.local.lcz_ts import lcz_ts

# 1a. Download (or load from cache) the LCZ map for Berlin
lcz_map = lcz_get_map(city="Berlin")
print("LCZ map:", lcz_map)

# 1b. Load the reference station dataset used across all local/ notebooks
df = pd.read_csv("https://raw.githubusercontent.com/ByMaxAnjos/LCZ4py/main/lcz_data_berlin.csv")
print(df.shape)
df.head()

06:28:36 - LCZ4py._internal._lcz_map_engine - INFO - Geocoding 'Berlin' via async HTTP...


06:28:37 - httpx - INFO - HTTP Request: GET https://nominatim.openstreetmap.org/search?q=Berlin&format=geojson&limit=1&polygon_geojson=1 "HTTP/1.1 200 OK"


06:28:37 - LCZ4py._internal._lcz_map_engine - INFO - Cached boundary to /Users/co2map/.lcz4r_cache/study_area_berlin.arrow


06:28:38 - LCZ4py._internal._lcz_map_engine - INFO - Streaming COG and clipping to geometry...


06:28:41 - rasterio._env - WARNING - CPLE_IllegalArg in tmprxe4adxp.tif: BLOCKXSIZE can only be used with TILED=YES


06:28:41 - LCZ4py._internal._lcz_map_engine - INFO - Saved to clipped cache.


LCZ map: /Users/co2map/.lcz4r_cache/clipped_2ed1dcfe644b4c6b.tif


(386354, 6)


,Unnamed: 0,date,station,airT,Latitude,Longitude
0,806404,2019-01-01 00:00:00,airportthf,8.50000,52.467500,13.402100
1,806405,2019-01-01 00:00:00,airporttxl,7.90000,52.564400,13.308800
2,806406,2019-01-01 00:00:00,albrecht,8.04725,52.444594,13.348607
3,806407,2019-01-01 00:00:00,bamberger,8.27166,52.496494,13.337552
4,806408,2019-01-01 00:00:00,baruth,8.20000,52.061300,13.499700


`lcz_map` is a path to a clipped GeoTIFF (classes 1–17) covering Berlin's extent. `df` has one row per station-hour. Neither object encodes LCZ membership yet — that happens inside `lcz_ts` itself via a spatial join against `lcz_map`, using `Latitude`/`Longitude` to look up each station's pixel class.

## 2. Default view: `plot_type="basic_line"`

The simplest call groups by station, averages to the requested `time_freq`, and draws one line per station. `lcz_ts` appends the resolved LCZ class to each station's name in the legend (e.g. `tiergarten (14)`), so you can already see urban-form grouping without any extra faceting. Here we aggregate to daily means over the full record to keep the chart readable.

## Publication-ready time-series export with `style`

The plotting functions now support `style="default"`, `style="nature"`, `style="science"`, and `style="generic_bw"`. The journal presets adjust the font family, physical sizing in millimeters, DPI (300 for the publication-oriented presets), and colors, so the same `lcz_ts` plot can be exported as a manuscript-ready figure with one extra argument.

A minimal publication-ready example:

- `style="science"` for the time-series figure.
- `isave=True` with `save_extension="png"` or `"pdf"` to write the final figure.


In [ ]:
fig_line_pub = lcz_ts(
    lcz_map,
    data_frame=df,
    var="airT",
    station_id="station",
    time_freq="day",
    plot_type="basic_line",
    style="science",
    isave=True,
    save_extension="png",
    lang="en",
)
fig_line_pub


In [3]:
fig_line = lcz_ts(
    x=lcz_map, data_frame=df, var="airT", station_id="station",
    time_freq="day", plot_type="basic_line", lang="en",
    title="Berlin station air temperature — daily mean, by LCZ-labeled station",
)
fig_line.show()

**Reading this chart**: each trace is one station, labeled with its LCZ class in parentheses. Lines cluster around the shared seasonal cycle (cold Dec–Feb, warm Jun–Aug), but at any given date the *spread* between lines is the interesting part — that spread is largely explained by which LCZ each station sits in, which the next charts make explicit.

## 3. Seasonal facets: `by="season"`

Setting `by="season"` splits the figure into one subplot per season, each showing the same per-station lines restricted to that season's dates. This isolates whether the LCZ-driven spread in station temperatures is stable across the year or changes seasonally (e.g. UHI effects are often strongest in summer nights and weakest under winter snow cover).

In [4]:
fig_season = lcz_ts(
    x=lcz_map, data_frame=df, var="airT", station_id="station",
    time_freq="day", by="season", lang="en",
    title="Berlin station air temperature by season",
)
fig_season.show()

**Reading this chart**: compare the vertical spread of station lines within each seasonal subplot. A wider spread in summer than winter is the classic seasonal signature of urban thermal contrast — compact built LCZs retain and re-radiate solar heat while open/natural LCZs cool more efficiently, and that gap widens under strong summer insolation.

## 4. LCZ-faceted heatmap: `plot_type="heatmap"`, `facet_plot="lcz"`

The heatmap view is the most direct way to see the diurnal fingerprint of each LCZ class: one row per LCZ class, hour along the x-axis, color = mean temperature. Here we restrict to July 2019 (northern-hemisphere summer, when UHI contrasts are typically sharpest) and aggregate hourly, so the diurnal cycle is visible.

In [5]:
fig_heatmap = lcz_ts(
    x=lcz_map, data_frame=df, var="airT", station_id="station",
    time_freq="hour", plot_type="heatmap", facet_plot="lcz",
    year=[2019], month=[7], lang="en",
    title="Berlin hourly air temperature by LCZ class — July 2019",
)
fig_heatmap.show()

**Reading this chart**: each row is an LCZ class present among the Berlin stations. Look for rows that stay warm (brighter color) through the night hours — those are the compact/urban classes (LCZ 1–3, 5–6) retaining heat — versus rows that cool sharply after sunset, typically open or vegetated classes (LCZ 11–14). This night-time persistence of warmth in built classes is the direct visual evidence of the canopy urban heat island that later notebooks (02, 03) will quantify numerically.

## 5. Daylight-aware view: `by="daylight"`

`by="daylight"` keeps all stations on one unified time axis (like the basic line chart) but shades daytime windows in amber, computed per-date from the NOAA solar elevation formula using the mean station latitude/longitude. This makes it easy to visually separate day-driven warming from night-time behavior without splitting the chart into subplots — useful for spotting whether inter-station spread is a daytime or a nighttime phenomenon.

In [6]:
fig_daylight = lcz_ts(
    x=lcz_map, data_frame=df, var="airT", station_id="station",
    time_freq="hour", by="daylight",
    year=[2019], month=[7], lang="en",
    title="Berlin air temperature, July 2019 — daytime windows shaded",
)
fig_daylight.show()

**Reading this chart**: temperature rises during each amber (daytime) band and falls in the unshaded nighttime gaps. If the inter-station spread visibly narrows during the shaded daytime bands and widens in the unshaded nighttime gaps, that is direct evidence that the urban heat island in Berlin is predominantly a **nocturnal** phenomenon — the classic pattern for mid-latitude cities, where compact urban LCZs release stored daytime heat overnight while open/rural LCZs radiate freely to a clear sky.

## Conclusion

This notebook set up the shared `local/` data pipeline (Berlin `lcz_map` + `lcz_data_berlin.csv`) and showed how `lcz_ts` turns raw station observations into LCZ-stratified time series through a single spatial join, then visualizes them with line, seasonal-facet, LCZ-faceted heatmap, and daylight-shaded views. Every later `local/` notebook reuses this same `lcz_map` + `df` pair for consistency.

**Previous**: `notebooks/general/07_gridded_climate_environment` (gridded climate/environmental data — the last notebook of the raster-wide "general" series).

**Next**: `notebooks/local/02_temperature_anomalies` — using the same LCZ-labeled stations to compute per-station temperature anomalies against the spatial mean.